# Runner — ejecuta los notebooks caso por caso

> **Qué hace.** Ejecuta, uno a uno, los notebooks de **EDA** y **modelamiento** de cada caso (más el de setup) y guarda una **copia ejecutada con todos sus gráficos** en `notebooks/_ejecutados/`. Así, si te pierdes en el camino, obtienes de un solo paso todos los resultados —figuras, desempeño, desarrollo— sin ejecutar cada notebook a mano.

> **Cómo usar.** Ábrelo y ejecuta todo de arriba abajo.

## Uso en Google Colab

1. Sube el proyecto a Colab (o clónalo si tienes acceso al repositorio) y sitúate en su carpeta.
2. Ejecuta este notebook con **Entorno de ejecución → Ejecutar todo**.

No necesitas instalar Python 3.13 a mano: `uv` lo descarga y aísla. Tampoco necesitas subir los datos por separado —las 12 fuentes ya viven versionadas en `data/01_raw/`—. En local, basta con abrir el notebook (idealmente con `uv run jupyter lab`) y ejecutar todo.

## 1. Preparar el entorno

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

# Raiz del proyecto (funciona desde notebooks/ o desde la raiz).
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
assert (ROOT / 'pyproject.toml').exists(), 'Abre este notebook dentro del proyecto tostao-retail-ml'
print('Proyecto:', ROOT)

# uv gestiona Python 3.13 + dependencias de forma reproducible; se instala si falta (p. ej. en Colab).
if shutil.which('uv') is None:
    print('Instalando uv...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)

def sh(cmd):
    """Ejecuta un comando y transmite su salida al notebook."""
    print('>', ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    if proc.returncode:
        raise RuntimeError(f'Comando fallo (rc={proc.returncode}): ' + ' '.join(cmd))

In [ ]:
# Incluye el grupo dev (jupyter/nbconvert) para poder ejecutar los notebooks.
sh(['uv', 'sync', '--extra', 'caso_b', '--group', 'dev'])

## 2. Ejecutar los notebooks

Cada notebook se ejecuta con `nbconvert` en el entorno reproducible; la copia ejecutada (con salidas) se guarda en `notebooks/_ejecutados/`. El original del repositorio no se modifica.

In [ ]:
OUT = Path('notebooks/_ejecutados')
OUT.mkdir(parents=True, exist_ok=True)

NOTEBOOKS = [
    'notebooks/00_setup.ipynb',
    'notebooks/caso_a/01_eda.ipynb', 'notebooks/caso_a/02_modelamiento.ipynb',
    'notebooks/caso_b/01_eda.ipynb', 'notebooks/caso_b/02_modelamiento.ipynb',
    'notebooks/caso_c/01_eda.ipynb', 'notebooks/caso_c/02_modelamiento.ipynb',
]

resultados = []
for nb in NOTEBOOKS:
    salida = nb.replace('notebooks/', '').replace('/', '__')
    print(f'\n=== Ejecutando {nb} ===')
    try:
        sh(['uv', 'run', 'jupyter', 'nbconvert', '--to', 'notebook', '--execute', nb,
            '--output-dir', str(OUT), '--output', salida,
            '--ExecutePreprocessor.timeout=900'])
        resultados.append((nb, 'OK'))
    except Exception as e:
        resultados.append((nb, f'FALLO: {e}'))

## 3. Resumen

In [ ]:
print('Resultado de la ejecución:\n')
for nb, estado in resultados:
    print(f'  {estado:8s} {nb}')
print('\nCopias ejecutadas (con gráficos) en:', OUT.resolve())

## Conclusión

Cada notebook de EDA y de modelamiento quedó ejecutado con sus figuras y tablas en `notebooks/_ejecutados/`. Para la ejecución productiva de punta a punta (sin abrir notebook por notebook), usa `run_pipeline.ipynb`.